In [1]:
!nvidia-sm

/bin/bash: line 1: nvidia-sm: command not found


In [2]:
import os
HOME = os.getcwd()
print(HOME)

/content


In [3]:
!pip install ultralytics==8.2.103 -q

from IPython import display
display.clear_output()

import ultralytics
ultralytics.checks()

Ultralytics YOLOv8.2.103 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 41.3/112.6 GB disk)


In [1]:
!mkdir -p {HOME}/datasets-new
%cd {HOME}/datasets-new

!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="QCHnLILdmDft5Z2j2gLS")
project = rf.workspace("rtdetractivevision").project("tennis-player-detection-ailvl-yjr30")
version = project.version(1)
dataset = version.download("yolov8")


/content/{HOME}/datasets-new
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Tennis-Player-Detection-1 in yolov8:: 100%|██████████| 11564/11564 [00:01<00:00, 5957.42it/s]


In [ ]:
%cd {HOME}

!yolo task=detect mode=train model=yolov8n.yaml data={dataset.location}/data.yaml pretrained=yolov8n.pt epochs=35 imgsz=640 plots=True

[Errno 2] No such file or directory: '{HOME}'
/content/{HOME}/datasets-new
100% 6.25M/6.25M [00:00<00:00, 47.0MB/s]
Transferred 355/355 items from pretrained weights
New https://pypi.org/project/ultralytics/8.3.97 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.103 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.yaml, data=/content/{HOME}/datasets-new/Tennis-Player-Detection-1/data.yaml, epochs=35, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=yolov8n.pt, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, 

In [ ]:
%cd {HOME}
Image(filename=f'{HOME}/runs/detect/train/confusion_matrix.png', width=600)

In [ ]:
%cd {HOME}

!yolo task=detect mode=val model={HOME}/runs/detect/train/weights/best.pt data={dataset.location}/data.yaml

In [ ]:
%cd {HOME}

!yolo export model={HOME}/runs/detect/train/weights/best.pt format=onnx

# Convert ONNX model to TFLite

Install onnx2tf package

In [ ]:
!sudo apt-get -y update
!sudo apt-get -y install python3-pip
!sudo apt-get -y install python-is-python3
!wget https://github.com/PINTO0309/onnx2tf/releases/download/1.16.31/flatc.tar.gz \
  && tar -zxvf flatc.tar.gz \
  && sudo chmod +x flatc \
  && sudo mv flatc /usr/bin/
!pip install -U pip \
  && pip install tensorflow==2.17.0 \
  && pip install -U onnx==1.16.1 \
  && python -m pip install onnx_graphsurgeon \
        --index-url https://pypi.ngc.nvidia.com \
  && pip install -U onnxruntime==1.18.1 \
  && pip install -U onnxsim==0.4.33 \
  && pip install -U simple_onnx_processing_tools \
  && pip install -U onnx2tf \
  && pip install -U protobuf==3.20.3 \
  && pip install -U h5py==3.11.0 \
  && pip install -U psutil==5.9.5 \
  && pip install -U ml_dtypes==0.3.2 \
  && pip install -U tf-keras~=2.16 \
  && pip install flatbuffers>=23.5.26

In [ ]:
!onnx2tf -i /content/yolov8_human_det.onnx -b 1 # replace "/content/yolov8_human_det.onnx" to your own onnx file path

In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

Convert to FP16 tflite model:

In [ ]:
import tensorflow as tf
import pathlib
model_path = "/content/saved_model"
loaded_model = tf.saved_model.load(model_path)
tflite_models_dir = pathlib.Path("/content/")
converter = tf.lite.TFLiteConverter.from_saved_model(model_path)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model_quant = converter.convert()

tflite_model_fp16_file = tflite_models_dir/"yolov8n_model_quant_f16.tflite"
tflite_model_fp16_file.write_bytes(tflite_model_quant)

Convert to full integer quant (UINT8) model:

In [ ]:
import tensorflow as tf
import os
import numpy as np
import cv2
import pathlib

def letterbox(img, new_shape=(640, 640)):
    """Resizes and reshapes images while maintaining aspect ratio by adding padding, suitable for YOLO models."""
    shape = img.shape[:2]  # current shape [height, width]

    # Scale ratio (new / old)
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])

    # Compute padding
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
    dw, dh = (new_shape[1] - new_unpad[0]) / 2, (new_shape[0] - new_unpad[1]) / 2  # wh padding

    if shape[::-1] != new_unpad:  # resize
        img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(114, 114, 114))

    return img, (top / img.shape[0], left / img.shape[1])

model_path = "/content/saved_model"
tflite_models_dir = pathlib.Path("/content/")

converter = tf.lite.TFLiteConverter.from_saved_model(model_path)

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,  # enable TensorFlow Lite ops.
    tf.lite.OpsSet.SELECT_TF_OPS,  # enable TensorFlow ops.
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

samples_folder = "/content/gdrive/MyDrive/dataset/player-calibration"

def representative_dataset():
    # List all image files in the samples folder
    image_files = [file for file in os.listdir(samples_folder) if file.endswith(('.jpg', '.jpeg', '.png'))]
    print(len(image_files))
    # Resize each image to the required dimensions and yield as representative data
    for image_file in image_files:
        image_path = os.path.join(samples_folder, image_file)
        print(image_path)
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        img, pad = letterbox(image)
        img = img[..., ::-1][None]  # N,H,W,C for TFLite
        img = np.ascontiguousarray(img)
        img = img.astype(np.float32)
        print(img.shape)
        yield [img / 255]


converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
# converter.experimental_new_converter = True

converter.inference_input_type = tf.uint8 # or tf.uint8
converter.inference_output_type = tf.uint8

tflite_model_quant = converter.convert()

tflite_model_int8_file = tflite_models_dir/"yolov8n_quant_uint8.tflite"
tflite_model_int8_file.write_bytes(tflite_model_quant)